In [2]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import os
import pandas as pd
from scipy.special import gamma

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from skimage.measure import shannon_entropy

np.random.seed(42)

# ===============================
# PATHS
# ===============================

image_folder = "/content/drive/MyDrive/Colab Notebooks/low_quality_images"

output_he = "/content/drive/MyDrive/Colab Notebooks/HE+ICS+HYBRID/HE"
output_ics = "/content/drive/MyDrive/Colab Notebooks/HE+ICS+HYBRID/ICS"
output_hybrid = "/content/drive/MyDrive/Colab Notebooks/HE+ICS+HYBRID/HYBRID"

os.makedirs(output_he, exist_ok=True)
os.makedirs(output_ics, exist_ok=True)
os.makedirs(output_hybrid, exist_ok=True)

# ===============================
# METRICS (UPDATED)
# ===============================

def compute_metrics(original, processed):
    original = original.astype(np.float32)
    processed = processed.astype(np.float32)

    # Existing
    ssim_val = ssim(original, processed, data_range=255)
    psnr_val = psnr(original, processed, data_range=255)
    mse_val = mse(original, processed)

    # NEW
    mae_val = np.mean(np.abs(original - processed))
    entropy_val = shannon_entropy(processed)

    edges_orig = cv2.Canny(original.astype(np.uint8), 100, 200)
    edges_proc = cv2.Canny(processed.astype(np.uint8), 100, 200)

    if np.sum(edges_orig) == 0:
        epi_val = 0
    else:
        epi_val = np.sum(edges_orig & edges_proc) / np.sum(edges_orig)

    return ssim_val, psnr_val, mse_val, mae_val, entropy_val, epi_val

# ===============================
# SIGMOID TRANSFORM
# ===============================

def apply_transform(y, a, b):
    y = y.astype(np.float32)/255.0
    out = 1/(1+np.exp(-a*(y-b)))
    return np.clip(out*255,0,255).astype(np.uint8)

# ===============================
# FITNESS FUNCTION
# ===============================

def tenengrad(img):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    return np.mean(gx**2 + gy**2)

def entropy(img):
    hist = cv2.calcHist([img],[0],None,[256],[0,256])
    hist /= hist.sum()
    return -np.sum(hist * np.log(hist + 1e-10))

def get_fitness(img):
    def f(p):
        t = apply_transform(img,p[0],p[1])
        return 0.4*tenengrad(t)+0.3*np.var(t)+0.3*entropy(t)
    return f

# ===============================
# ICS CLASS
# ===============================

class ICS:
    def __init__(self,fitness,bounds,n=12,pa=0.25,beta=1.5,iters=15):
        self.fit=fitness
        self.bounds=np.array(bounds)
        self.n=n
        self.pa=pa
        self.beta=beta
        self.iters=iters
        self.dim=len(bounds)

        self.pop=np.random.uniform(self.bounds[:,0],self.bounds[:,1],(n,self.dim))
        self.fvals=np.array([self.fit(x) for x in self.pop])

    def levy(self):
        b=self.beta
        sigma=(gamma(1+b)*np.sin(np.pi*b/2)/(gamma((1+b)/2)*b*2**((b-1)/2)))**(1/b)
        u=np.random.normal(0,sigma,self.dim)
        v=np.random.normal(0,1,self.dim)
        return u/(np.abs(v)**(1/b))

    def clip(self,x):
        return np.clip(x,self.bounds[:,0],self.bounds[:,1])

    def run(self):
        for _ in range(self.iters):
            for i in range(self.n):
                new=self.clip(self.pop[i]+0.01*self.levy())
                fnew=self.fit(new)
                j=np.random.randint(self.n)
                if fnew>self.fvals[j]:
                    self.pop[j]=new
                    self.fvals[j]=fnew

            for i in range(self.n):
                if np.random.rand()<self.pa:
                    self.pop[i]=np.random.uniform(self.bounds[:,0],self.bounds[:,1],self.dim)
                    self.fvals[i]=self.fit(self.pop[i])

        return self.pop[np.argmax(self.fvals)]

# ===============================
# LOAD IMAGES
# ===============================

files = [f for f in os.listdir(image_folder)
         if f.lower().endswith((".jpg",".png",".jpeg"))][:25]

records = []
bounds=[(0.5,4.0),(0.2,0.8)]

# ===============================
# MAIN LOOP
# ===============================

for file in files:

    img = cv2.imread(os.path.join(image_folder,file))
    gray = cv2.resize(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),(512,512))

    # HE
    he = cv2.equalizeHist(gray)
    cv2.imwrite(os.path.join(output_he,file),he)

    # ICS
    ics = ICS(get_fitness(gray),bounds)
    best = ics.run()
    enhanced = apply_transform(gray,best[0],best[1])
    cv2.imwrite(os.path.join(output_ics,file),enhanced)

    # HYBRID
    ics_he = ICS(get_fitness(he),bounds)
    best_he = ics_he.run()
    hybrid = apply_transform(he,best_he[0],best_he[1])
    cv2.imwrite(os.path.join(output_hybrid,file),hybrid)

    # METRICS
    he_vals = compute_metrics(gray, he)
    ics_vals = compute_metrics(gray, enhanced)
    hyb_vals = compute_metrics(gray, hybrid)

    records.append([
        file,
        *he_vals,
        *ics_vals,
        *hyb_vals
    ])

    print("Processed:", file)

# ===============================
# SAVE CSV
# ===============================

df = pd.DataFrame(records, columns=[
    "Image",
    "HE_SSIM","HE_PSNR","HE_MSE","HE_MAE","HE_Entropy","HE_EPI",
    "ICS_SSIM","ICS_PSNR","ICS_MSE","ICS_MAE","ICS_Entropy","ICS_EPI",
    "HYB_SSIM","HYB_PSNR","HYB_MSE","HYB_MAE","HYB_Entropy","HYB_EPI"
])

df.to_csv("/content/drive/MyDrive/Colab Notebooks/HE+ICS+HYBRID/final_results.csv",index=False)

print("\nAverage Metrics:\n",df.mean(numeric_only=True))

Mounted at /content/drive
Processed: 00 (74).jpg
Processed: 00 (139).jpg
Processed: 00 (84).jpg
Processed: 00 (137).jpg
Processed: 00 (59).jpg
Processed: 00 (112).jpg
Processed: 00 (79).jpg
Processed: 00 (57).jpg
Processed: 00 (132).jpg
Processed: 00 (81).jpg
Processed: 00 (111).jpg
Processed: 00 (109).jpg
Processed: 00 (106).jpg
Processed: 00 (116).jpg
Processed: 00 (131).jpg
Processed: 00 (78).jpg
Processed: 00 (77).jpg
Processed: 00 (141).jpg
Processed: 00 (138).jpg
Processed: 00 (80).jpg
Processed: 00 (54).jpg
Processed: 00 (113).jpg
Processed: 00 (134).jpg
Processed: 00 (133).jpg
Processed: 00 (107).jpg

Average Metrics:
 HE_SSIM          0.859194
HE_PSNR         19.362997
HE_MSE         884.304543
HE_MAE          23.908045
HE_Entropy       7.317644
HE_EPI           0.693179
ICS_SSIM         0.947097
ICS_PSNR        22.646479
ICS_MSE        546.083945
ICS_MAE         19.671995
ICS_Entropy      7.145902
ICS_EPI          0.801311
HYB_SSIM         0.883886
HYB_PSNR        20.950586
H